In [1]:
# ============================================================
# ส่วนที่ 0: ตั้งค่าพื้นฐาน (CONFIG) - แก้ path ให้ตรงกับของจริงบน Kaggle
# ============================================================
import os, glob, warnings, pickle
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")

# ---- พิกัด virtual buoy 15 จุด (คง ID เดิม 1-12, 14, 17, 19) ----
POINTS = {
    1:  (8.2899, 104.4261), 2:  (10.4942, 102.0876), 3:  (8.4334, 101.6216),
    4:  (10.5101, 101.1532), 5:  (8.3111, 102.5559), 6:  (8.2739, 100.6867),
    7:  (6.3708, 104.8937), 8:  (6.3307, 103.9583), 9:  (10.5486, 100.2207),
    10: (8.4341, 103.4907), 11: (12.3024, 100.6907), 12: (6.3751, 103.0239),
    14: (12.5765, 96.9441), 17: (8.2497, 97.8824), 19: (10.5285, 97.4159),
}  # value = (lat, lon)

CONFIG = {
    "ERA5_ROOT": "/kaggle/input/datasets/iwarinchuensaen/era5-2012-2026-8vars",
    "L3_DIR":    "/kaggle/input/datasets/pornphanphuatana/swh-l3",
    "GFS_DIR":   "/kaggle/input/datasets/iwarinchuensaen/gfswave-thailand/netcdf",
    "OUT_DIR":   "/kaggle/working",
    "TIME_TOL_MIN": 90,
    "COAST_BUFFER_KM": 25,
}
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

# ---- Checkpoint helper: กัน kernel restart ต้องรันซ้ำตั้งแต่ต้น ----
CKPT_DIR = f"{CONFIG['OUT_DIR']}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_ckpt(obj, name):
    with open(f"{CKPT_DIR}/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)
    print(f"[checkpoint] บันทึก {name}")

def load_ckpt(name):
    path = f"{CKPT_DIR}/{name}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            obj = pickle.load(f)
        print(f"[checkpoint] โหลด {name} จากไฟล์ที่บันทึกไว้")
        return obj
    return None

def rmse_score(y_true, y_pred):
    """แทน mean_squared_error(..., squared=False) ที่ถูกถอดออกใน sklearn รุ่นใหม่"""
    from sklearn.metrics import mean_squared_error
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [22]:
# ============================================================
# ส่วนที่ 1: โหลดข้อมูลดิบทั้ง 3 แหล่ง
# ============================================================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def load_era5():
    """
    หมายเหตุ (ก.ย. 2026): ตัด wave_direction ออกชั่วคราว - ไฟล์ที่มีอยู่ (mvd_YYYY.nc)
    เปิดแล้วพบว่าเป็น 'inss' (turbulent surface stress) ไม่ใช่ mean wave direction จริง
    ต้องไปดึงใหม่จาก CDS API ด้วย short name 'mwd' แล้วค่อยเพิ่มกลับเข้า pipeline
    """
    var_folder = {
        "swh": "wave_swh2012-2026_3hr/SWH",
        "mwp": "wave_mean_period2012-2026_3hr/MWP",
        "u10": "ERA5_u10", "v10": "ERA5_v10",
        "t2m": "ERA5_t2m", "sst": "ERA5_sst", "fg10": "ERA5_fg10",
    }
    candidate_names = {
        "swh": ["swh"], "mwp": ["mwp"],
        "u10": ["u10"], "v10": ["v10"],
        "t2m": ["t2m"], "sst": ["sst"], "fg10": ["fg10"],
    }
    ds_list = []
    for var, folder in var_folder.items():
        base = os.path.join(CONFIG["ERA5_ROOT"], folder)
        files = sorted(glob.glob(os.path.join(base, "**", "*.nc"), recursive=True))
        if not files:
            print(f"[คำเตือน] ไม่พบไฟล์ {var} ใน {base}")
            continue
        print(f"  {var}: พบ {len(files)} ไฟล์ เช่น {os.path.basename(files[0])}")
        da = xr.open_mfdataset(files, combine="by_coords")
        if "valid_time" in da.coords:
            da = da.rename({"valid_time": "time"})
        actual_var = next((v for v in candidate_names[var] if v in da.variables), None)
        if actual_var is None:
            print(f"    [คำเตือน] ไม่เจอ {var} ตัวแปรที่มีจริง: {list(da.data_vars)}")
            continue
        if actual_var != var:
            da = da.rename({actual_var: var})
        ds_list.append(da[[var]])
    era5 = xr.merge(ds_list, compat="override", join="inner")
    print("ERA5 loaded:", dict(era5.sizes))
    return era5


def load_l3():
    files = sorted(glob.glob(os.path.join(CONFIG["L3_DIR"], "**", "*.nc"), recursive=True))
    if not files:
        raise FileNotFoundError(f"ไม่พบไฟล์ L3 ใน {CONFIG['L3_DIR']}")
    frames = []
    for f in files:
        ds = xr.open_dataset(f)
        var_name = "VAVH" if "VAVH" in ds.variables else [v for v in ds.variables if "vavh" in v.lower()][0]
        df = ds[[var_name]].to_dataframe().reset_index().rename(columns={var_name: "l3_swh"})
        df["mission"] = os.path.basename(f).split("_")[0]
        frames.append(df[["time", "latitude", "longitude", "l3_swh", "mission"]])
        ds.close()
    l3 = pd.concat(frames, ignore_index=True).dropna(subset=["l3_swh"])
    l3 = l3.rename(columns={"latitude": "lat", "longitude": "lon"})
    print(f"L3 loaded: {len(l3):,} จุด จาก {l3['mission'].nunique()} mission(s)")
    return l3


def inspect_gfs():
    files = sorted(glob.glob(os.path.join(CONFIG["GFS_DIR"], "*.nc")))
    print(f"พบไฟล์ GFS-Wave {len(files)} ไฟล์ (ใช้จริงในขั้นตอน Model B)")
    return files


era5_ds = load_ckpt("era5_ds")
if era5_ds is None:
    era5_ds = load_era5()
    save_ckpt(era5_ds, "era5_ds")

l3_df = load_ckpt("l3_df")
if l3_df is None:
    l3_df = load_l3()
    save_ckpt(l3_df, "l3_df")

gfs_files = inspect_gfs()

[checkpoint] โหลด era5_ds จากไฟล์ที่บันทึกไว้
L3 loaded: 540,219 จุด จาก 2 mission(s)
[checkpoint] บันทึก l3_df
พบไฟล์ GFS-Wave 98 ไฟล์ (ใช้จริงในขั้นตอน Model B)


In [23]:
# ============================================================
# ส่วนที่ 2: QC + Collocation ระหว่าง L3 กับ ERA5 ที่ 15 จุด virtual buoy
# ============================================================
def collocate_buoy(buoy_id, lat0, lon0, l3_df, era5_ds,
                    radius_km=CONFIG["COAST_BUFFER_KM"]*2,
                    time_tol_min=CONFIG["TIME_TOL_MIN"]):
    dist = haversine_km(l3_df["lat"].values, l3_df["lon"].values, lat0, lon0)
    sub = l3_df.loc[dist <= radius_km].copy()
    if sub.empty:
        return pd.DataFrame()
    sub["dist_km"] = dist[dist <= radius_km]

    era5_point = era5_ds.sel(latitude=lat0, longitude=lon0, method="nearest")
    era5_point_df = era5_point.to_dataframe().reset_index()

    sub = sub.sort_values("time")
    era5_point_df = era5_point_df.sort_values("time")
    matched = pd.merge_asof(
        sub, era5_point_df, on="time", direction="nearest",
        tolerance=pd.Timedelta(minutes=time_tol_min),
    ).dropna(subset=["swh"])
    matched["buoy_id"], matched["buoy_lat"], matched["buoy_lon"] = buoy_id, lat0, lon0
    return matched


colloc = load_ckpt("colloc")
if colloc is None:
    all_matched = [collocate_buoy(bid, lat0, lon0, l3_df, era5_ds) for bid, (lat0, lon0) in POINTS.items()]
    for (bid, _), m in zip(POINTS.items(), all_matched):
        print(f"buoy {bid}: จับคู่ได้ {len(m)} จุด")
    colloc = pd.concat(all_matched, ignore_index=True)

    colloc["wind_speed"] = np.sqrt(colloc["u10"]**2 + colloc["v10"]**2)
    colloc["wind_dir"] = (np.degrees(np.arctan2(colloc["u10"], colloc["v10"])) + 360) % 360
    doy = colloc["time"].dt.dayofyear
    colloc["doy_sin"] = np.sin(2*np.pi*doy/365.25)
    colloc["doy_cos"] = np.cos(2*np.pi*doy/365.25)
    colloc["residual"] = colloc["l3_swh"] - colloc["swh"]

    save_ckpt(colloc, "colloc")
    colloc.to_csv(f"{CONFIG['OUT_DIR']}/colloc_l3_era5.csv", index=False)

print("collocation shape:", colloc.shape)

buoy 1: จับคู่ได้ 7693 จุด
buoy 2: จับคู่ได้ 7403 จุด
buoy 3: จับคู่ได้ 7279 จุด
buoy 4: จับคู่ได้ 7480 จุด
buoy 5: จับคู่ได้ 7335 จุด
buoy 6: จับคู่ได้ 7494 จุด
buoy 7: จับคู่ได้ 7178 จุด
buoy 8: จับคู่ได้ 7378 จุด
buoy 9: จับคู่ได้ 7307 จุด
buoy 10: จับคู่ได้ 7227 จุด
buoy 11: จับคู่ได้ 7368 จุด
buoy 12: จับคู่ได้ 6996 จุด
buoy 14: จับคู่ได้ 6683 จุด
buoy 17: จับคู่ได้ 6571 จุด
buoy 19: จับคู่ได้ 6541 จุด
[checkpoint] บันทึก colloc
collocation shape: (107933, 25)


In [24]:
# ============================================================
# QC ก่อนตัดสินใจเรื่อง L3 aggregation
# ============================================================

print("=" * 80)
print("L3 RAW DATA QC")
print("=" * 80)

print("Raw L3 observations :", len(l3_df))
print("Collocated rows      :", len(colloc))

print("\nL3 time range:")
print(l3_df["time"].min(), "→", l3_df["time"].max())

print("\nBy year:")
display(
    l3_df
    .assign(year=l3_df["time"].dt.year)
    .groupby("year")
    .size()
    .rename("n")
    .to_frame()
)

print("\nBy mission:")
display(
    l3_df
    .groupby("mission")
    .size()
    .sort_values(ascending=False)
    .rename("n")
    .to_frame()
)

print("\nUnique source files:")
if "source_file" in l3_df.columns:
    display(
        l3_df
        .groupby("source_file")
        .size()
        .sort_values(ascending=False)
        .rename("n")
        .to_frame()
    )

print("\nL3 observations per buoy after current collocation:")
display(
    colloc
    .groupby("buoy_id")
    .size()
    .rename("n")
    .to_frame()
)

L3 RAW DATA QC
Raw L3 observations : 540219
Collocated rows      : 107933

L3 time range:
2021-01-01 03:46:32 → 2026-08-27 03:38:35

By year:


,n
year,
2021,28781
2022,26427
2023,132217
2024,128830
2025,133849
2026,90115



By mission:


,n
mission,
Sentinel-3A.nc,443187
VAVH,97032



Unique source files:

L3 observations per buoy after current collocation:


,n
buoy_id,
1,7693
2,7403
3,7279
4,7480
5,7335
6,7494
7,7178
8,7378
9,7307


In [25]:
# ============================================================
# ส่วนที่ 3: Model A - LightGBM ทำนาย residual (L3 - ERA5)
# ============================================================
import lightgbm as lgb
import joblib

FEATURES = ["swh", "mwp", "wind_speed", "wind_dir", "t2m", "sst", "fg10",
            "buoy_lat", "buoy_lon", "doy_sin", "doy_cos"]
# ***ห้ามใส่ 'mission' เป็น feature*** ใช้แค่ตอนแบ่ง validation เท่านั้น

df = colloc.dropna(subset=FEATURES + ["residual"]).copy()
df["year"] = df["time"].dt.year
test_year = df["year"].max()
train_df, test_df = df[df["year"] < test_year], df[df["year"] == test_year]
X_train, y_train = train_df[FEATURES], train_df["residual"]
X_test, y_test = test_df[FEATURES], test_df["residual"]

model_A = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
model_A.fit(X_train, y_train, eval_set=[(X_test, y_test)],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

# --- Leave-one-mission-out ---
missions = train_df["mission"].unique()
if len(missions) > 1:
    print("\n--- Leave-one-mission-out validation ---")
    for m in missions:
        tr, va = train_df[train_df["mission"] != m], train_df[train_df["mission"] == m]
        if va.empty: continue
        m_model = lgb.LGBMRegressor(**model_A.get_params())
        m_model.fit(tr[FEATURES], tr["residual"])
        rmse = rmse_score(va["residual"], m_model.predict(va[FEATURES]))
        print(f"  ตัด mission {m:10s} ออก -> RMSE(residual) = {rmse:.3f} m (n={len(va)})")
else:
    print("มี mission เดียวในข้อมูล L3 ตอนนี้ - ข้าม leave-one-mission-out ไปก่อน")

# --- ประเมินผลบนปีล่าสุด ---
pred_residual = model_A.predict(X_test)
corrected_swh = X_test["swh"].values + pred_residual
observed_l3 = test_df["l3_swh"].values

bias = np.mean(corrected_swh - observed_l3)
rmse = rmse_score(observed_l3, corrected_swh)
raw_rmse = rmse_score(observed_l3, X_test["swh"].values)
si = rmse / np.mean(observed_l3)

print(f"\nraw ERA5   RMSE = {raw_rmse:.3f} m")
print(f"corrected  RMSE = {rmse:.3f} m   bias = {bias:+.3f} m   SI = {si:.3f}")
for p in [90, 95, 99]:
    print(f"P{p}: L3={np.percentile(observed_l3,p):.2f} m | "
          f"raw ERA5={np.percentile(X_test['swh'].values,p):.2f} m | "
          f"corrected={np.percentile(corrected_swh,p):.2f} m")

joblib.dump(model_A, f"{CONFIG['OUT_DIR']}/model_A_lightgbm.pkl")
save_ckpt(model_A, "model_A")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2315
[LightGBM] [Info] Number of data points in the train set: 90533, number of used features: 11
[LightGBM] [Info] Start training from score 0.187329
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0349221
Early stopping, best iteration is:
[114]	valid_0's l2: 0.0348624

--- Leave-one-mission-out validation ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001538 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2315
[LightGBM] [Info] Number of data points in the train set: 15960, number of used features: 11
[LightGBM] [Info] Start training from score 0.188211
  ตัด mission Sentinel-3A.nc ออก -> RMSE(residual) = 0

In [27]:
# ============================================================
# ส่วนที่ 4: Apply Model A ทั้ง ERA5 series -> historical virtual buoy + climatology
# ============================================================
def build_full_era5_features(era5_ds, buoy_id, lat0, lon0):
    pt = era5_ds.sel(latitude=lat0, longitude=lon0, method="nearest").to_dataframe().reset_index()
    pt["wind_speed"] = np.sqrt(pt["u10"]**2 + pt["v10"]**2)
    pt["wind_dir"] = (np.degrees(np.arctan2(pt["u10"], pt["v10"])) + 360) % 360
    doy = pt["time"].dt.dayofyear
    pt["doy_sin"] = np.sin(2*np.pi*doy/365.25)
    pt["doy_cos"] = np.cos(2*np.pi*doy/365.25)
    pt["buoy_id"], pt["buoy_lat"], pt["buoy_lon"] = buoy_id, lat0, lon0
    return pt

historical = load_ckpt("historical_corrected")
if historical is None:
    frames = []
    for bid, (lat0, lon0) in POINTS.items():
        pt = build_full_era5_features(era5_ds, bid, lat0, lon0).dropna(subset=FEATURES)
        pt["residual_pred"] = model_A.predict(pt[FEATURES])
        pt["swh_corrected"] = pt["swh"] + pt["residual_pred"]
        frames.append(pt[["time", "buoy_id", "buoy_lat", "buoy_lon",
                           "swh", "swh_corrected", "wind_speed", "wind_dir", "mwp"]])
        print(f"buoy {bid}: แก้ไขแล้ว {len(pt):,} จุดเวลา")
    historical = pd.concat(frames, ignore_index=True)
    save_ckpt(historical, "historical_corrected")
    historical.to_csv(f"{CONFIG['OUT_DIR']}/historical_virtual_buoy.csv", index=False)

print("historical corrected shape:", historical.shape)


def compute_climatology(historical, window_days=15, percentiles=(90, 95, 99), min_n=30):
    df = historical.copy()
    df["doy"] = df["time"].dt.dayofyear
    rows = []
    for bid, g in df.groupby("buoy_id"):
        doy_vals, swh_vals = g["doy"].values, g["swh_corrected"].values
        for d in range(1, 367):
            diff = np.abs(doy_vals - d)
            diff = np.minimum(diff, 366 - diff)          # จัดการรอยต่อปลาย/ต้นปี
            mask = diff <= window_days
            if mask.sum() < min_n:
                continue
            row = {"buoy_id": bid, "doy": d, "n_samples": int(mask.sum())}
            for p in percentiles:
                row[f"P{p}"] = float(np.percentile(swh_vals[mask], p))
            rows.append(row)
    return pd.DataFrame(rows)

climatology = load_ckpt("climatology")
if climatology is None:
    climatology = compute_climatology(historical)
    save_ckpt(climatology, "climatology")
    climatology.to_csv(f"{CONFIG['OUT_DIR']}/climatology_p90_95_99.csv", index=False)

print("climatology shape:", climatology.shape)
print(climatology.groupby("buoy_id")[["P90", "P95", "P99"]].mean().round(2))

buoy 1: แก้ไขแล้ว 42,732 จุดเวลา
buoy 2: แก้ไขแล้ว 42,732 จุดเวลา
buoy 3: แก้ไขแล้ว 42,732 จุดเวลา
buoy 4: แก้ไขแล้ว 42,732 จุดเวลา
buoy 5: แก้ไขแล้ว 42,732 จุดเวลา
buoy 6: แก้ไขแล้ว 42,732 จุดเวลา
buoy 7: แก้ไขแล้ว 42,732 จุดเวลา
buoy 8: แก้ไขแล้ว 42,732 จุดเวลา
buoy 9: แก้ไขแล้ว 42,732 จุดเวลา
buoy 10: แก้ไขแล้ว 42,732 จุดเวลา
buoy 11: แก้ไขแล้ว 42,732 จุดเวลา
buoy 12: แก้ไขแล้ว 42,732 จุดเวลา
buoy 14: แก้ไขแล้ว 42,732 จุดเวลา
buoy 17: แก้ไขแล้ว 42,732 จุดเวลา
buoy 19: แก้ไขแล้ว 42,732 จุดเวลา
[checkpoint] บันทึก historical_corrected
historical corrected shape: (640980, 9)
[checkpoint] บันทึก climatology
climatology shape: (5490, 6)
          P90   P95   P99
buoy_id                  
1        1.26  1.41  1.66
2        1.21  1.38  1.73
3        1.27  1.45  1.79
4        1.27  1.44  1.84
5        1.34  1.53  1.88
6        1.04  1.18  1.46
7        1.68  1.91  2.35
8        1.54  1.75  2.18
9        1.17  1.34  1.73
10       1.34  1.52  1.84
11       0.96  1.07  1.30
12       1.35  1.54

In [29]:
# ============================================================
# ส่วนที่ 5: ตรวจจับ Anomaly - (a) Percentile exceedance (b) Isolation Forest
# ============================================================
historical["doy"] = historical["time"].dt.dayofyear
hist_flagged = historical.merge(climatology, on=["buoy_id", "doy"], how="left")
for p in [90, 95, 99]:
    hist_flagged[f"exceed_p{p}"] = hist_flagged["swh_corrected"] > hist_flagged[f"P{p}"]
    rate = hist_flagged[f"exceed_p{p}"].mean() * 100
    print(f"สัดส่วนที่เกิน P{p}: {rate:.2f}% (คาดหวัง ~{100-p}%)")

from sklearn.ensemble import IsolationForest
from sklearn.metrics import cohen_kappa_score

def add_if_features(df):
    df = df.sort_values(["buoy_id", "time"]).copy()
    df["deseasonal_anom"] = df["swh_corrected"] - df.groupby("doy")["swh_corrected"].transform("mean")
    df["delta_6h"] = df.groupby("buoy_id")["swh_corrected"].diff(2)    # 3hr step x2 = 6h
    df["delta_24h"] = df.groupby("buoy_id")["swh_corrected"].diff(8)   # 3hr step x8 = 24h
    return df

hist_if = add_if_features(hist_flagged)
IF_FEATURES = ["swh_corrected", "deseasonal_anom", "wind_speed", "mwp", "delta_6h", "delta_24h"]
hist_if_clean = hist_if.dropna(subset=IF_FEATURES).copy()

TARGET_RATE = 0.01   # อัตรา flag ให้เท่ากับ P99 เพื่อเทียบกันยุติธรรม
iso = IsolationForest(n_estimators=300, contamination=TARGET_RATE, random_state=42)
scores = -iso.fit(hist_if_clean[IF_FEATURES]).score_samples(hist_if_clean[IF_FEATURES])
hist_if_clean["if_score"] = scores
score_thresh = np.percentile(scores, 100 * (1 - TARGET_RATE))
median_by_doy = hist_if_clean.groupby("doy")["swh_corrected"].transform("median")
hist_if_clean["if_anomaly_high"] = (scores >= score_thresh) & (hist_if_clean["swh_corrected"] > median_by_doy)

print("\n--- Sensitivity analysis ---")
for rate in [0.005, 0.01, 0.02, 0.05]:
    iso_r = IsolationForest(n_estimators=300, contamination=rate, random_state=42)
    s = -iso_r.fit(hist_if_clean[IF_FEATURES]).score_samples(hist_if_clean[IF_FEATURES])
    thr = np.percentile(s, 100 * (1 - rate))
    n_flag = (s >= thr).sum()
    print(f"contamination={rate:.3f} -> flag {n_flag:,} จุด ({n_flag/len(s)*100:.2f}%)")

common = hist_flagged.merge(
    hist_if_clean[["time", "buoy_id", "if_anomaly_high"]], on=["time", "buoy_id"], how="inner"
)
kappa = cohen_kappa_score(common["exceed_p99"], common["if_anomaly_high"])
print(f"\nCohen's kappa (P99 vs Isolation Forest): {kappa:.3f}")

save_ckpt(hist_if_clean, "hist_if_clean")

สัดส่วนที่เกิน P90: 9.63% (คาดหวัง ~10%)
สัดส่วนที่เกิน P95: 4.76% (คาดหวัง ~5%)
สัดส่วนที่เกิน P99: 0.95% (คาดหวัง ~1%)

--- Sensitivity analysis ---
contamination=0.005 -> flag 3,205 จุด (0.50%)
contamination=0.010 -> flag 6,409 จุด (1.00%)
contamination=0.020 -> flag 12,818 จุด (2.00%)
contamination=0.050 -> flag 32,043 จุด (5.00%)

Cohen's kappa (P99 vs Isolation Forest): 0.280
[checkpoint] บันทึก hist_if_clean


In [30]:
# เช็คว่าจุดที่ flag ที่ 0.5% เป็น subset ของจุดที่ flag ที่ 1%, 2%, 5% หรือไม่ (ควรเป็น nested)
results = {}
for rate in [0.005, 0.01, 0.02, 0.05]:
    iso_r = IsolationForest(n_estimators=300, contamination=rate, random_state=42)
    s = -iso_r.fit(hist_if_clean[IF_FEATURES]).score_samples(hist_if_clean[IF_FEATURES])
    thr = np.percentile(s, 100 * (1 - rate))
    results[rate] = set(hist_if_clean.index[s >= thr])

base = results[0.005]
for rate in [0.01, 0.02, 0.05]:
    overlap = len(base & results[rate]) / len(base) * 100
    print(f"จุดที่ flag ที่ 0.5% ยังอยู่ในกลุ่ม {rate*100:.0f}% หรือไม่: {overlap:.1f}%")

จุดที่ flag ที่ 0.5% ยังอยู่ในกลุ่ม 1% หรือไม่: 100.0%
จุดที่ flag ที่ 0.5% ยังอยู่ในกลุ่ม 2% หรือไม่: 100.0%
จุดที่ flag ที่ 0.5% ยังอยู่ในกลุ่ม 5% หรือไม่: 100.0%


In [31]:
common = hist_flagged.merge(
    hist_if_clean[["time", "buoy_id", "if_anomaly_high", "deseasonal_anom", "wind_speed", "delta_24h"]],
    on=["time", "buoy_id"], how="inner"
)

tab = pd.crosstab(common["exceed_p99"], common["if_anomaly_high"],
                   rownames=["Percentile P99"], colnames=["Isolation Forest"])
print(tab)

for bid, g in common.groupby("buoy_id"):
    if g["exceed_p99"].sum() < 5:
        continue
    k = cohen_kappa_score(g["exceed_p99"], g["if_anomaly_high"])
    print(f"buoy {bid}: kappa = {k:.3f} (n exceed_p99={g['exceed_p99'].sum()})")
    

Isolation Forest   False  True 
Percentile P99                 
False             630176   4618
True                4275   1791
buoy 1: kappa = 0.176 (n exceed_p99=402)
buoy 2: kappa = 0.412 (n exceed_p99=409)
buoy 3: kappa = 0.350 (n exceed_p99=408)
buoy 4: kappa = 0.500 (n exceed_p99=410)
buoy 5: kappa = 0.402 (n exceed_p99=394)
buoy 6: kappa = 0.270 (n exceed_p99=397)
buoy 7: kappa = 0.144 (n exceed_p99=406)
buoy 8: kappa = 0.174 (n exceed_p99=410)
buoy 9: kappa = 0.379 (n exceed_p99=421)
buoy 10: kappa = 0.350 (n exceed_p99=405)
buoy 11: kappa = 0.135 (n exceed_p99=412)
buoy 12: kappa = 0.253 (n exceed_p99=409)
buoy 14: kappa = 0.345 (n exceed_p99=409)
buoy 17: kappa = 0.300 (n exceed_p99=388)
buoy 19: kappa = 0.370 (n exceed_p99=386)


In [32]:
# ============================================================
# ส่วนที่ 6: Model B - แก้ไข GFS-Wave forecast
# ============================================================
def inspect_gfs_file(path):
    ds = xr.open_dataset(path)
    print(f"ไฟล์: {os.path.basename(path)}")
    print(f"มิติ: {dict(ds.sizes)}")
    print(f"coords: {list(ds.coords)}")
    print(f"ตัวแปร: {list(ds.data_vars)}")
    for v in list(ds.data_vars)[:10]:
        print(f"  {v}: long_name='{ds[v].attrs.get('long_name','?')}' units='{ds[v].attrs.get('units','?')}'")
    ds.close()

# --- รันบรรทัดนี้ก่อน แล้วดู output ว่าตรงกับที่คาดไหม ---
inspect_gfs_file(gfs_files[0])

ไฟล์: gfswave_20260904_00z.nc
มิติ: {'latitude': 37, 'longitude': 40}
coords: ['time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
ตัวแปร: ['ws', 'wdir', 'swh', 'perpw', 'dirpw']
  ws: long_name='Wind speed' units='m s**-1'
  wdir: long_name='Wind direction' units='Degree true'
  swh: long_name='Significant height of combined wind waves and swell' units='m'
  perpw: long_name='Primary wave mean period' units='s'
  dirpw: long_name='Primary wave direction' units='Degree true'


# test before add sentimeter


In [6]:
import shutil

# checkpoint เหล่านี้สร้างจาก l3_df รุ่นเก่า (อาจมีข้อมูลซ้ำ) ถ้าไม่ย้ายออก load_ckpt จะดึงของเก่ากลับมาเงียบๆ
STALE = ["l3_df", "colloc", "model_A", "historical_corrected", "climatology",
         "hist_if_clean", "hist_if_v2_clean", "model_B", "lead_error_std"]
stale_dir = f"{CKPT_DIR}/stale_before_l3_rewrite"
os.makedirs(stale_dir, exist_ok=True)
for n in STALE:
    p = f"{CKPT_DIR}/{n}.pkl"
    if os.path.exists(p):
        shutil.move(p, f"{stale_dir}/{n}.pkl")
        print("ย้ายออก:", n)

# ลบตัวแปรเก่าในหน่วยความจำ (ชื่อ l3_df ถูกเลิกใช้ ขั้นต่อไปจะใช้ l3_raw / l3_clean)
for v in ["l3_df", "colloc", "model_A", "historical", "climatology", "hist_if_clean"]:
    globals().pop(v, None)

ย้ายออก: l3_df
ย้ายออก: colloc
ย้ายออก: model_A
ย้ายออก: historical_corrected
ย้ายออก: climatology


In [7]:
import re

# ---------- 1) หา mission จาก metadata ก่อน แล้วค่อย fallback ไป path ----------
MISSION_PATTERNS = [
    (r"sentinel[\s_\-]?3[\s_\-]?a|\bs3a\b", "Sentinel-3A"),
    (r"sentinel[\s_\-]?3[\s_\-]?b|\bs3b\b", "Sentinel-3B"),
    (r"sentinel[\s_\-]?6", "Sentinel-6A"),
    (r"jason[\s_\-]?3", "Jason-3"),
    (r"saral|altika", "SARAL"),
    (r"cryosat", "CryoSat-2"),
    (r"cfosat", "CFOSAT"),
    (r"swot", "SWOT"),
    (r"hy[\s_\-]?2b", "HY-2B"),
    (r"hy[\s_\-]?2c", "HY-2C"),
]
ATTR_KEYS = ["platform", "platform_name", "mission", "satellite", "source",
             "title", "product_name", "dataset_id", "id"]

def match_missions(text):
    text = text.lower()
    return [name for pat, name in MISSION_PATTERNS if re.search(pat, text)]

def resolve_mission(attrs, rel_path):
    attr_text = " ".join(str(v) for k, v in attrs.items() if k.lower() in ATTR_KEYS)
    for text, src in [(attr_text, "attrs"), (rel_path, "path")]:
        found = match_missions(text)
        if len(found) == 1:
            return found[0], src
        if len(found) > 1:
            return "MULTI:" + "+".join(found), src
    return "UNKNOWN", "none"


# ---------- 2) inventory: เปิดทุกไฟล์แบบเบา ๆ ดูโครงสร้างก่อนโหลดจริง ----------
def scan_l3_inventory(files):
    rows = []
    for f in files:
        rel = os.path.relpath(f, CONFIG["L3_DIR"])
        try:
            ds = xr.open_dataset(f)
        except Exception as e:
            rows.append({"source_file": rel, "error": str(e)[:80]})
            continue
        mission, msrc = resolve_mission(ds.attrs, rel)
        tname = next((c for c in ds.variables if c.lower() == "time"), None)
        tmin = tmax = n = None
        if tname is not None:
            t = pd.to_datetime(ds[tname].values)
            tmin, tmax, n = t.min(), t.max(), len(t)
        rows.append({
            "source_file": rel, "mission": mission, "mission_source": msrc,
            "n_time": n, "t_min": tmin, "t_max": tmax,
            "data_vars": ",".join(ds.data_vars),
            "attr_platform": str(ds.attrs.get("platform", ds.attrs.get("platform_name", "-")))[:30],
            "attr_title": str(ds.attrs.get("title", "-"))[:50],
        })
        ds.close()
    return pd.DataFrame(rows)


# ---------- 3) loader ใหม่: มี source_file / mission จริง / คอลัมน์ quality ถ้ามี ----------
def make_keys(df):
    """key_fine  = เวลาระดับวินาที + lat/lon ทศนิยม 3 ตำแหน่ง (~100 ม.)  -> จับข้อมูลซ้ำแบบเป๊ะ
       key_coarse = เวลาปัด 5 วินาที + lat/lon ทศนิยม 2 ตำแหน่ง (~1 กม.) -> จับข้อมูลซ้ำที่ precision ต่างกันนิดหน่อย
       (ตั้งใจไม่ใส่ mission ใน key เพราะไฟล์ซ้ำอาจติดป้าย mission ต่างกัน)"""
    t = df["time"].values.astype("datetime64[s]").astype("int64")
    lat = df["lat"].values.astype("float64")
    lon = df["lon"].values.astype("float64")
    fine = pd.util.hash_pandas_object(
        pd.DataFrame({"t": t, "la": np.round(lat, 3), "lo": np.round(lon, 3)}), index=False).values
    coarse = pd.util.hash_pandas_object(
        pd.DataFrame({"t": (t // 5) * 5, "la": np.round(lat, 2), "lo": np.round(lon, 2)}), index=False).values
    return df.assign(key_fine=fine, key_coarse=coarse)


def load_l3_v2(files, inventory):
    inv = inventory.set_index("source_file")
    frames = []
    for f in files:
        rel = os.path.relpath(f, CONFIG["L3_DIR"])
        if rel not in inv.index or "error" in inv.columns and pd.notna(inv.loc[rel].get("error", np.nan)):
            print(f"[ข้าม] เปิดไฟล์ไม่ได้: {rel}")
            continue
        ds = xr.open_dataset(f).reset_coords()
        df = ds.to_dataframe().reset_index()
        ds.close()
        cols = {c.lower(): c for c in df.columns}
        tcol = cols.get("time")
        latc = cols.get("latitude") or cols.get("lat")
        lonc = cols.get("longitude") or cols.get("lon")
        swhc = cols.get("vavh")
        if None in (tcol, latc, lonc, swhc):
            print(f"[ข้าม] {rel}: ไม่พบ time/lat/lon/VAVH ครบ (คอลัมน์ที่มี: {list(df.columns)[:10]})")
            continue
        out = pd.DataFrame({"time": pd.to_datetime(df[tcol].values),
                            "lat": df[latc].values, "lon": df[lonc].values,
                            "l3_swh": df[swhc].values})
        for lc, orig in cols.items():          # เก็บคอลัมน์ quality/unfiltered/wind ไว้ใช้ตอน 2A
            if lc != "vavh" and any(k in lc for k in ["vavh", "wind", "qual", "flag"]):
                out[lc] = df[orig].values
        out["mission"] = inv.loc[rel, "mission"]
        out["mission_source"] = inv.loc[rel, "mission_source"]
        out["source_file"] = rel
        if inv.loc[rel, "n_time"] is not None and len(out) != inv.loc[rel, "n_time"]:
            print(f"[คำเตือน] {rel}: แถวที่โหลด {len(out)} != n_time {inv.loc[rel, 'n_time']}")
        frames.append(out)
    l3 = pd.concat(frames, ignore_index=True)
    return make_keys(l3)


l3_files = sorted(glob.glob(os.path.join(CONFIG["L3_DIR"], "**", "*.nc"), recursive=True))
print(f"พบไฟล์ L3 ทั้งหมด {len(l3_files)} ไฟล์")

l3_inventory = scan_l3_inventory(l3_files)
pd.set_option("display.max_colwidth", 60, "display.width", 200)
display(l3_inventory)

# ดู global attrs ของไฟล์ตัวอย่าง (ช่วยตัดสินว่า VAVH_YYYY.nc เป็น product อะไร)
for f in l3_files[:1] + [p for p in l3_files if "VAVH" in os.path.basename(p)][:1]:
    print("\n--- global attrs:", os.path.relpath(f, CONFIG["L3_DIR"]))
    with xr.open_dataset(f) as _ds:
        for k, v in list(_ds.attrs.items())[:25]:
            print(f"  {k}: {str(v)[:100]}")

l3_raw = load_ckpt("l3_raw")
if l3_raw is None:
    l3_raw = load_l3_v2(l3_files, l3_inventory)
    save_ckpt(l3_raw, "l3_raw")
print("\nl3_raw:", l3_raw.shape, "| คอลัมน์:", list(l3_raw.columns))

พบไฟล์ L3 ทั้งหมด 17 ไฟล์


,source_file,mission,mission_source,n_time,t_min,t_max,data_vars,attr_platform,attr_title
0,2021/SWH_L3_2021/Sentinel-3A.nc,Sentinel-3A,path,152331,2021-01-01 03:46:32,2026-08-26 15:06:18,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
1,2023/SWH_L3_2023/Sentinel-3A.nc,Sentinel-3A,path,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
2,2023/SWH_L3_2023_(1)/Sentinel-3A.nc,Sentinel-3A,path,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
3,2023/SWH_L3_2023_(2)/Sentinel-3A.nc,Sentinel-3A,path,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
4,2023/VAVH_2023.nc,UNKNOWN,none,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
5,2024/SWH_L3_2024/Sentinel-3A.nc,Sentinel-3A,path,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
6,2024/SWH_L3_2024_(1)/Sentinel-3A.nc,Sentinel-3A,path,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
7,2024/SWH_L3_2024_(2)/Sentinel-3A.nc,Sentinel-3A,path,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
8,2024/VAVH_2024.nc,UNKNOWN,none,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s
9,2025/SWH_L3_2025_(1)/Sentinel-3A.nc,Sentinel-3A,path,26755,2025-01-01 15:11:57,2025-12-30 15:02:01,VAVH,-,Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s



--- global attrs: 2021/SWH_L3_2021/Sentinel-3A.nc
  institution: CLS, CNES, EUMETSAT
  doi: https://doi.org/10.48670/moi-00179
  time_coverage_start: 2021-01-01T03:46:32Z
  time_coverage_end: 2026-08-26T15:06:18Z
  last_date_observation: 2026-08-26T15:06:18Z
  geospatial_lat_min: 5.000024
  geospatial_lat_max: 15.999925
  last_latitude_observation: 12.5733
  geospatial_lon_min: 95.00004
  geospatial_lon_max: 104.99989
  last_longitude_observation: 101.59677
  geospatial_vertical_min: -0.0
  geospatial_vertical_max: -0.0
  references: http://marine.copernicus.eu
  license: https://marine.copernicus.eu/user-corner/service-commitments-and-licence
  download_date: 2026-08-27T15:59:45.899355
  copernicusmarine_toolbox_version: 2.4.1
  product_id: WAVE_GLO_PHY_SWH_L3_NRT_014_001
  dataset_id: cmems_obs-wave_glo_phy-swh_nrt_s3a-l3_PT1S
  Conventions: CF-1.9 ACDD-1.3
  title: Subset of dataset cmems_obs-wave_glo_phy-swh_nrt_s3a-l3_PT1S with Copernicus Marine Toolbox
  contact: servicedesk.cme

In [8]:
def provenance_qc(l3):
    print("=" * 80); print("L3 PROVENANCE QC"); print("=" * 80)
    print(f"แถวทั้งหมด: {len(l3):,}")
    print(f"lat: {l3['lat'].min():.2f}..{l3['lat'].max():.2f} | "
          f"lon: {l3['lon'].min():.2f}..{l3['lon'].max():.2f}  (เช็คว่า lon เป็น -180..180 หรือ 0..360)")

    l3 = l3.assign(year=l3["time"].dt.year)

    print("\n[1] จำนวนแถว แยกตาม mission x ปี")
    display(l3.pivot_table(index="mission", columns="year", values="l3_swh", aggfunc="size", fill_value=0))

    print("\n[2] สรุปรายไฟล์ + ความถูกต้องพื้นฐาน")
    per_file = l3.groupby("source_file").agg(
        mission=("mission", "first"), n_rows=("l3_swh", "size"),
        t_min=("time", "min"), t_max=("time", "max"),
        swh_min=("l3_swh", "min"), swh_mean=("l3_swh", "mean"), swh_max=("l3_swh", "max"),
        n_nan_swh=("l3_swh", lambda s: int(s.isna().sum())),
        n_swh_le0=("l3_swh", lambda s: int((s <= 0).sum())),
        n_swh_gt15=("l3_swh", lambda s: int((s > 15).sum())),
        n_dup_within=("key_fine", lambda s: int(s.duplicated().sum())),
    )
    display(per_file.round(2))

    print("\n[3] ข้อมูลซ้ำข้ามไฟล์ (key = เวลา + lat/lon โดยไม่สนป้าย mission)")
    files_per_key = l3.groupby("key_fine")["source_file"].nunique()
    shared_keys = files_per_key[files_per_key > 1].index
    per_file_share = l3.assign(shared=l3["key_fine"].isin(shared_keys)).groupby("source_file").agg(
        n_rows=("shared", "size"), n_shared=("shared", "sum"))
    per_file_share["pct_shared"] = (per_file_share["n_shared"] / per_file_share["n_rows"] * 100).round(1)
    display(per_file_share)

    if len(shared_keys):
        sh = l3[l3["key_fine"].isin(shared_keys)][["key_fine", "source_file", "l3_swh"]]
        print("\nคู่ไฟล์ที่ใช้ key ซ้ำกันมากที่สุด:")
        pairs = sh.merge(sh, on="key_fine")
        pairs = pairs[pairs["source_file_x"] < pairs["source_file_y"]]
        display(pairs.groupby(["source_file_x", "source_file_y"]).size()
                .sort_values(ascending=False).head(20).rename("n_shared_keys").to_frame())
        agg = sh.groupby("key_fine")["l3_swh"].agg(["max", "min"])
        spread = (agg["max"] - agg["min"]).dropna()
        print(f"\nค่า SWH ต่างกันแค่ไหนในจุดที่ซ้ำ (max-min ต่อ key): "
              f"เท่ากันเป๊ะ {(spread == 0).mean()*100:.1f}% | "
              f"p50={spread.quantile(.5):.4f} p95={spread.quantile(.95):.4f} max={spread.max():.4f} m")
    else:
        print("ไม่พบ key ซ้ำข้ามไฟล์")

    print("\n[4] ประมาณจำนวนข้อมูลอิสระจริง")
    n_fine, n_coarse = l3["key_fine"].nunique(), l3["key_coarse"].nunique()
    print(f"แถวดิบ           : {len(l3):,}")
    print(f"unique key_fine  : {n_fine:,}  (ซ้ำแบบเป๊ะ {len(l3)-n_fine:,} แถว)")
    print(f"unique key_coarse: {n_coarse:,}  (ซ้ำเพิ่มเมื่อผ่อน precision {n_fine-n_coarse:,} แถว)")
    uniq = l3.drop_duplicates("key_fine")
    print("\nหลังตัดซ้ำแบบเป๊ะ แยกตามปี x mission:")
    display(uniq.pivot_table(index="mission", columns="year", values="l3_swh", aggfunc="size", fill_value=0))

provenance_qc(l3_raw)

L3 PROVENANCE QC
แถวทั้งหมด: 540,219
lat: 5.00..16.00 | lon: 95.00..105.00  (เช็คว่า lon เป็น -180..180 หรือ 0..360)

[1] จำนวนแถว แยกตาม mission x ปี


year,2021,2022,2023,2024,2025,2026
mission,,,,,,
Sentinel-3A,28781,26427,105801,103064,107094,72020
UNKNOWN,0,0,26416,25766,26755,18095



[2] สรุปรายไฟล์ + ความถูกต้องพื้นฐาน


,mission,n_rows,t_min,t_max,swh_min,swh_mean,swh_max,n_nan_swh,n_swh_le0,n_swh_gt15,n_dup_within
source_file,,,,,,,,,,,
2021/SWH_L3_2021/Sentinel-3A.nc,Sentinel-3A,152331,2021-01-01 03:46:32,2026-08-26 15:06:18,0.03,0.96,3.78,0,0,0,0
2023/SWH_L3_2023/Sentinel-3A.nc,Sentinel-3A,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,0.16,0.96,3.54,0,0,0,0
2023/SWH_L3_2023_(1)/Sentinel-3A.nc,Sentinel-3A,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,0.16,0.96,3.54,0,0,0,0
2023/SWH_L3_2023_(2)/Sentinel-3A.nc,Sentinel-3A,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,0.16,0.96,3.54,0,0,0,0
2023/VAVH_2023.nc,UNKNOWN,26416,2023-01-01 03:21:20,2023-12-30 03:12:05,0.16,0.96,3.54,0,0,0,0
2024/SWH_L3_2024/Sentinel-3A.nc,Sentinel-3A,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,0.19,0.95,2.96,0,0,0,0
2024/SWH_L3_2024_(1)/Sentinel-3A.nc,Sentinel-3A,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,0.19,0.95,2.96,0,0,0,0
2024/SWH_L3_2024_(2)/Sentinel-3A.nc,Sentinel-3A,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,0.19,0.95,2.96,0,0,0,0
2024/VAVH_2024.nc,UNKNOWN,25766,2024-01-01 15:00:12,2024-12-30 03:23:28,0.19,0.95,2.96,0,0,0,0



[3] ข้อมูลซ้ำข้ามไฟล์ (key = เวลา + lat/lon โดยไม่สนป้าย mission)


,n_rows,n_shared,pct_shared
source_file,,,
2021/SWH_L3_2021/Sentinel-3A.nc,152331,96912,63.6
2023/SWH_L3_2023/Sentinel-3A.nc,26416,26416,100.0
2023/SWH_L3_2023_(1)/Sentinel-3A.nc,26416,26416,100.0
2023/SWH_L3_2023_(2)/Sentinel-3A.nc,26416,26416,100.0
2023/VAVH_2023.nc,26416,26416,100.0
2024/SWH_L3_2024/Sentinel-3A.nc,25766,25766,100.0
2024/SWH_L3_2024_(1)/Sentinel-3A.nc,25766,25766,100.0
2024/SWH_L3_2024_(2)/Sentinel-3A.nc,25766,25766,100.0
2024/VAVH_2024.nc,25766,25766,100.0



คู่ไฟล์ที่ใช้ key ซ้ำกันมากที่สุด:


n_shared_keys
source_file_x                       source_file_y                                     
2025/SWH_L3_2025_(2)/Sentinel-3A.nc 2025/VAVH_2025.nc                            26755
2025/SWH_L3_2025_(1)/Sentinel-3A.nc 2025/VAVH_2025/Sentinel-3A.nc                26755
                                    2025/VAVH_2025.nc                            26755
                                    2025/SWH_L3_2025_(2)/Sentinel-3A.nc          26755
2021/SWH_L3_2021/Sentinel-3A.nc     2025/VAVH_2025/Sentinel-3A.nc                26755
                                    2025/VAVH_2025.nc                            26755
                                    2025/SWH_L3_2025_(2)/Sentinel-3A.nc          26755
                                    2025/SWH_L3_2025_(1)/Sentinel-3A.nc          26755
2025/SWH_L3_2025_(2)/Sentinel-3A.nc 2025/VAVH_2025/Sentinel-3A.nc                26755
2025/VAVH_2025.nc                   2025/VAVH_2025/Sentinel-3A.nc                26755
2023/SWH_L3_2023_(1)/Sentinel-3A.nc 2023/VAVH_2023.nc                            26416
2023/SWH_L3_2023_(2)/Sentinel-3A.nc 2023/VAVH_2023.nc                            26416
2021/SWH_L3_2021/Sentinel-3A.nc     2023/VAVH_2023.nc                            26416
                                    2023/SWH_L3_2023_(2)/Sentinel-3A.nc          26416
                                    2023/SWH_L3_2023_(1)/Sentinel-3A.nc          26416
                                    2023/SWH_L3_2023/Sentinel-3A.nc              26416
2023/SWH_L3_2023_(1)/Sentinel-3A.nc 2023/SWH_L3_2023_(2)/Sentinel-3A.nc          26416
2023/SWH_L3_2023/Sentinel-3A.nc     2023/VAVH_2023.nc                            26416
                                    2023/SWH_L3_2023_(2)/Sentinel-3A.nc          26416
                                    2023/SWH_L3_2023_(1)/Sentinel-3A.nc          26416


ค่า SWH ต่างกันแค่ไหนในจุดที่ซ้ำ (max-min ต่อ key): เท่ากันเป๊ะ 100.0% | p50=0.0000 p95=0.0000 max=0.0000 m

[4] ประมาณจำนวนข้อมูลอิสระจริง
แถวดิบ           : 540,219
unique key_fine  : 152,451  (ซ้ำแบบเป๊ะ 387,768 แถว)
unique key_coarse: 152,451  (ซ้ำเพิ่มเมื่อผ่อน precision 0 แถว)

หลังตัดซ้ำแบบเป๊ะ แยกตามปี x mission:


year,2021,2022,2023,2024,2025,2026
mission,,,,,,
Sentinel-3A,28781,26427,26553,25766,26829,18095


In [9]:
l3_dedup = load_ckpt("l3_dedup")
if l3_dedup is None:
    l3_dedup = l3_raw.drop_duplicates("key_fine").sort_values("time").reset_index(drop=True)
    save_ckpt(l3_dedup, "l3_dedup")

print(f"ก่อนตัดซ้ำ: {len(l3_raw):,} | หลังตัดซ้ำ: {len(l3_dedup):,}")
print(l3_dedup.groupby(l3_dedup["time"].dt.year).size().rename("n"))

[checkpoint] บันทึก l3_dedup
ก่อนตัดซ้ำ: 540,219 | หลังตัดซ้ำ: 152,451
time
2021    28781
2022    26427
2023    26553
2024    25766
2025    26829
2026    18095
Name: n, dtype: int64


In [10]:
def aggregate_passes(l3, points, radius_km, gap_minutes=10):
    """
    สำหรับแต่ละ buoy: เลือกจุด L3 ในรัศมี -> เรียงเวลา -> แยกเป็น pass ตาม time gap
    -> เฉลี่ยแต่ละ pass เป็น 1 representative observation
    """
    rows = []
    for bid, (lat0, lon0) in points.items():
        dist = haversine_km(l3["lat"].values, l3["lon"].values, lat0, lon0)
        sub = l3.loc[dist <= radius_km].sort_values("time").copy()
        if sub.empty:
            print(f"buoy {bid}: ไม่มีจุด L3 ในรัศมี {radius_km} กม.")
            continue
        gap = sub["time"].diff().dt.total_seconds() / 60
        pass_id = (gap > gap_minutes).cumsum()
        sub["pass_id"] = pass_id
        agg = sub.groupby("pass_id").agg(
            time=("time", "mean"), lat=("lat", "mean"), lon=("lon", "mean"),
            l3_swh=("l3_swh", "mean"), l3_swh_sd=("l3_swh", "std"),
            n_points=("l3_swh", "size"), mission=("mission", "first"),
        ).reset_index(drop=True)
        agg["buoy_id"], agg["buoy_lat"], agg["buoy_lon"] = bid, lat0, lon0
        rows.append(agg)
        print(f"buoy {bid}: {len(sub):,} จุดดิบ -> {len(agg)} pass")
    return pd.concat(rows, ignore_index=True)

PASS_RADIUS_KM = 55   # ~ครึ่งหนึ่งของ grid ERA5 (0.5 deg ~ 55 กม.) - ตั้งจากขนาด grid ไม่ใช่จาก COAST_BUFFER

l3_passes = load_ckpt("l3_passes")
if l3_passes is None:
    l3_passes = aggregate_passes(l3_dedup, POINTS, radius_km=PASS_RADIUS_KM)
    save_ckpt(l3_passes, "l3_passes")

print("\nจำนวน pass ต่อ buoy:")
display(l3_passes.groupby("buoy_id").size().rename("n_pass").to_frame())
print("\nจำนวน pass ต่อปี:")
display(l3_passes.groupby(l3_passes["time"].dt.year).size().rename("n_pass").to_frame())
print(f"\nn_points ต่อ pass: median={l3_passes['n_points'].median():.0f}, "
      f"min={l3_passes['n_points'].min()}, max={l3_passes['n_points'].max()}")

buoy 1: 2,309 จุดดิบ -> 150 pass
buoy 2: 2,284 จุดดิบ -> 147 pass
buoy 3: 2,247 จุดดิบ -> 147 pass
buoy 4: 2,285 จุดดิบ -> 150 pass
buoy 5: 2,304 จุดดิบ -> 148 pass
buoy 6: 2,290 จุดดิบ -> 149 pass
buoy 7: 2,175 จุดดิบ -> 148 pass
buoy 8: 2,267 จุดดิบ -> 146 pass
buoy 9: 2,252 จุดดิบ -> 147 pass
buoy 10: 2,255 จุดดิบ -> 145 pass
buoy 11: 2,273 จุดดิบ -> 151 pass
buoy 12: 2,169 จุดดิบ -> 145 pass
buoy 14: 2,095 จุดดิบ -> 140 pass
buoy 17: 2,062 จุดดิบ -> 139 pass
buoy 19: 2,102 จุดดิบ -> 142 pass
[checkpoint] บันทึก l3_passes

จำนวน pass ต่อ buoy:


,n_pass
buoy_id,
1,150
2,147
3,147
4,150
5,148
6,149
7,148
8,146
9,147



จำนวน pass ต่อปี:


,n_pass
time,
2021,395
2022,375
2023,385
2024,385
2025,394
2026,260



n_points ต่อ pass: median=16, min=1, max=17


In [17]:
def collocate_bilinear(l3_passes, era5_ds, time_tol_hours=3):
    lat_da = xr.DataArray(l3_passes["lat"].values, dims="obs")
    lon_da = xr.DataArray(l3_passes["lon"].values, dims="obs")
    time_da = xr.DataArray(l3_passes["time"].values, dims="obs")

    t_min, t_max = era5_ds["time"].values.min(), era5_ds["time"].values.max()
    lat_min, lat_max = float(era5_ds["latitude"].min()), float(era5_ds["latitude"].max())
    lon_min, lon_max = float(era5_ds["longitude"].min()), float(era5_ds["longitude"].max())
    in_bounds = (
        (l3_passes["time"].values >= t_min) & (l3_passes["time"].values <= t_max) &
        (l3_passes["lat"].between(min(lat_min, lat_max), max(lat_min, lat_max))) &
        (l3_passes["lon"].between(min(lon_min, lon_max), max(lon_min, lon_max)))
    )
    dropped = (~in_bounds).sum()
    if dropped:
        print(f"[คำเตือน] ตัด {dropped} pass ที่อยู่นอกขอบเขต ERA5 ออกก่อน interpolate")
    lp = l3_passes.loc[in_bounds].reset_index(drop=True)

    lat_da = xr.DataArray(lp["lat"].values, dims="obs")
    lon_da = xr.DataArray(lp["lon"].values, dims="obs")
    time_da = xr.DataArray(lp["time"].values, dims="obs")

    interp = era5_ds.interp(latitude=lat_da, longitude=lon_da, time=time_da, method="linear")
    era5_at_obs = interp.to_dataframe().reset_index(drop=True)

    # ตัดคอลัมน์ coordinate ที่ชื่อซ้ำกับ lp ออกก่อน concat (มีอยู่จริงคือ time/lat/lon/obs)
    dup_cols = [c for c in ["time", "latitude", "longitude", "lat", "lon", "obs"] if c in era5_at_obs.columns]
    era5_at_obs = era5_at_obs.drop(columns=dup_cols)

    out = pd.concat([lp.reset_index(drop=True), era5_at_obs], axis=1)
    return out.dropna(subset=["swh"])

In [16]:
print("colloc_v2" in dir())   # ถ้า True แปลว่ามีตัวแปรพังค้างอยู่ในหน่วยความจำ ลบทิ้งก่อน
if "colloc_v2" in dir():
    del colloc_v2

True


In [18]:
colloc_v2 = load_ckpt("colloc_v2")
if colloc_v2 is None:
    colloc_v2 = collocate_bilinear(l3_passes, era5_ds)

    colloc_v2["wind_speed"] = np.sqrt(colloc_v2["u10"]**2 + colloc_v2["v10"]**2)
    wd = np.arctan2(colloc_v2["u10"], colloc_v2["v10"])
    colloc_v2["wind_dir_sin"], colloc_v2["wind_dir_cos"] = np.sin(wd), np.cos(wd)
    doy = colloc_v2["time"].dt.dayofyear
    colloc_v2["doy_sin"], colloc_v2["doy_cos"] = np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)
    colloc_v2["residual"] = colloc_v2["l3_swh"] - colloc_v2["swh"]

    save_ckpt(colloc_v2, "colloc_v2")
    colloc_v2.to_csv(f"{CONFIG['OUT_DIR']}/colloc_v2_bilinear.csv", index=False)

print("colloc_v2 shape:", colloc_v2.shape)
print("จำนวนต่อ buoy:")
display(colloc_v2.groupby("buoy_id").size().rename("n").to_frame())

[คำเตือน] ตัด 13 pass ที่อยู่นอกขอบเขต ERA5 ออกก่อน interpolate
[checkpoint] บันทึก colloc_v2
colloc_v2 shape: (2177, 24)
จำนวนต่อ buoy:


,n
buoy_id,
1,148
2,145
3,145
4,149
5,146
6,148
7,148
8,146
9,146


In [19]:
print("จำนวน pass ต่อปี:")
display(l3_passes.groupby(l3_passes["time"].dt.year).size().rename("n_pass").to_frame())

print("\ncolloc_v2 shape:", colloc_v2.shape)
print("\nการกระจายของ l3_swh ใน colloc_v2 (ดูว่าเก็บคลื่นสูงได้ครอบคลุมแค่ไหน):")
print(colloc_v2["l3_swh"].describe())
for p in [50, 75, 90, 95, 99]:
    print(f"P{p}: {np.percentile(colloc_v2['l3_swh'], p):.2f} m")

จำนวน pass ต่อปี:


,n_pass
time,
2021,395
2022,375
2023,385
2024,385
2025,394
2026,260



colloc_v2 shape: (2177, 24)

การกระจายของ l3_swh ใน colloc_v2 (ดูว่าเก็บคลื่นสูงได้ครอบคลุมแค่ไหน):
count    2177.000000
mean        0.899121
std         0.457029
min         0.311667
25%         0.587500
50%         0.757177
75%         1.077000
max         3.629500
Name: l3_swh, dtype: float64
P50: 0.76 m
P75: 1.08 m
P90: 1.53 m
P95: 1.86 m
P99: 2.45 m
